# 07.0 Ablations

## Notebook overview
This notebook reruns selected SSL downstream settings while changing one design choice at a time so the robustness of the main findings can be assessed without rebuilding the full project pipeline.

## Purpose
- load the shared split manifest and leakage checks from notebook 01
- load the saved SimCLR encoder checkpoints from notebook 04
- rerun controlled SSL downstream comparisons for augmentation strength, embedding layer, and PatchCore coreset ratio
- export compact ablation tables and a summary figure that can be reused directly in the GitHub repo and final report

## Process
1. Resolve the dataset path and verify that the required outputs from notebooks 01 and 04 already exist.
2. Rebuild only the SSL downstream runtimes needed for the ablation study.
3. Evaluate the requested ablation settings category by category.
4. Save category-level and mean summary tables plus a compact ablation figure.


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Optional dependency install
#------------------------------------------------------------------------------
try:
    import faiss
    print("faiss already available")
except Exception:
    !pip -q install -U faiss-cpu


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import random
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.metrics import roc_auc_score, average_precision_score

import faiss
from IPython.display import display


# Print a clean banner so long notebook output is easier to scan.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create the parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read a JSON file from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return file size in MB if the path exists.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Keep deterministic behaviour consistent with the earlier notebooks.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Build a stable category-specific seed from text.
def stable_seed_from_text(seed, text):
    key = f"{seed}_{text}".encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


# Reset tracked peak CUDA memory before one fit stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Read the tracked peak CUDA memory in MB.
def get_peak_gpu_memory_mb():
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan


print_banner("Environment")
print("Python        :", sys.version.split()[0])
print("Torch         :", torch.__version__)
print("Torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    print(f"GPU {gpu_idx} name   :", torch.cuda.get_device_name(gpu_idx))


# 2.0 Run settings

## Purpose
- define the ablation settings in one place before any runtime building starts
- keep the notebook portable across Kaggle and local use
- save tables and figures into a clean folder structure that matches the other hardened notebooks


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "07_ablations"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16

BACKBONE_NAME = "resnet18"
FEATURE_LAYER_NAME_MAIN = "layer3"
CORESET_RATIO_MAIN = 0.05
MAX_TRAIN_PATCHES = 120_000
PATCH_SCORE_TOPK = 200
PADIM_DIM = 100
PADIM_EPS = 1e-4

LAYER_ABL_VALUES = ["layer2", "layer3", "layer4"]
CORESET_ABL_VALUES = [0.02, 0.05, 0.10]

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
NOTEBOOK_04_ROOT = WORK_ROOT / "04_simclr_pretraining" / RUN_MODE

SPLIT_MANIFEST_PATH = NOTEBOOK_01_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = NOTEBOOK_01_ROOT / "json" / "leakage_report.json"

SSL_CHECKPOINTS_DIR = NOTEBOOK_04_ROOT / "checkpoints"
TABLE_SIMCLR_SUMMARY_DEP_PATH = NOTEBOOK_04_ROOT / "tables" / "table_simclr_summary.csv"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
ABLATION_SETTINGS_PATH = JSON_DIR / "ablation_settings.json"
AVAILABLE_SSL_CHECKPOINTS_PATH = JSON_DIR / "available_ssl_checkpoints.json"

TABLE_ABLATION_CATEGORY_PATH = TABLES_DIR / "table_ablation_category.csv"
TABLE_ABLATION_MEAN_PATH = TABLES_DIR / "table_ablation_mean.csv"
TABLE_ABLATION_FULL_PATH = TABLES_DIR / "table_ablation_full.csv"

FIG_ABLATION_SUMMARY_PATH = FIGURES_DIR / "fig_ablation_summary.png"

RUN_METADATA = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": GPU_COUNT,
    "timestamp": datetime.now().isoformat(),
}
save_json_overwrite(RUN_METADATA, RUN_METADATA_PATH)

ABLATION_SETTINGS = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "categories_active": CATEGORIES_ACTIVE,
    "image_size": IMG_SIZE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "backbone_name": BACKBONE_NAME,
    "feature_layer_name_main": FEATURE_LAYER_NAME_MAIN,
    "coreset_ratio_main": CORESET_RATIO_MAIN,
    "layer_ablation_values": LAYER_ABL_VALUES,
    "coreset_ablation_values": CORESET_ABL_VALUES,
    "patchcore": {
        "max_train_patches": MAX_TRAIN_PATCHES,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
    "padim": {
        "dim": PADIM_DIM,
        "eps": PADIM_EPS,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
}
save_json_overwrite(ABLATION_SETTINGS, ABLATION_SETTINGS_PATH)

print_banner("Run configuration")
print("RUN_MODE           :", RUN_MODE)
print("SEED               :", SEED)
print("DEVICE             :", DEVICE)
print("GPU_COUNT          :", GPU_COUNT)
print("FEATURE_LAYER_MAIN :", FEATURE_LAYER_NAME_MAIN)
print("CORESET_RATIO_MAIN :", CORESET_RATIO_MAIN)
print("LAYER_ABL_VALUES   :", LAYER_ABL_VALUES)
print("CORESET_ABL_VALUES :", CORESET_ABL_VALUES)
print("NOTEBOOK_ROOT      :", NOTEBOOK_ROOT)


# 3.0 Dataset and prior artefacts

## Purpose
- resolve the dataset path in a way that works in Kaggle or a local clone
- load the split manifest and leakage report created in notebook 01
- verify that the required SimCLR outputs from notebook 04 exist before any ablation runs start


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve dataset path and load prior notebook artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

required_paths = [
    SPLIT_MANIFEST_PATH,
    LEAKAGE_REPORT_PATH,
]
missing_required = [str(path) for path in required_paths if not Path(path).exists()]
if missing_required:
    raise FileNotFoundError(
        "One or more dependency artefacts are missing. Run notebook 01 first.\n"
        + "\n".join(missing_required)
    )

split_manifest = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)

leakage_failures = []
for key, value in leakage_report.items():
    if isinstance(value, (int, float)) and value != 0:
        leakage_failures.append((key, value))
    elif isinstance(value, list) and len(value) > 0:
        leakage_failures.append((key, f"{len(value)} records"))

if leakage_failures:
    raise RuntimeError(
        "Notebook 01 reported non-zero leakage checks. Fix notebook 01 before continuing.\n"
        + "\n".join([f"{k}: {v}" for k, v in leakage_failures])
    )

ssl_ckpt_map = {
    "mild": SSL_CHECKPOINTS_DIR / "simclr_mild_encoder.pt",
    "strong": SSL_CHECKPOINTS_DIR / "simclr_strong_encoder.pt",
}
available_ssl_ckpt_map = {tag: str(path) for tag, path in ssl_ckpt_map.items() if path.exists()}
available_ssl_tags = list(available_ssl_ckpt_map.keys())
save_json_overwrite(available_ssl_ckpt_map, AVAILABLE_SSL_CHECKPOINTS_PATH)

if len(available_ssl_tags) == 0:
    raise FileNotFoundError(
        f"No SSL encoder checkpoints were found in {SSL_CHECKPOINTS_DIR}. "
        "Run 04_simclr_pretraining.ipynb first."
    )

table_simclr_summary = None
if TABLE_SIMCLR_SUMMARY_DEP_PATH.exists():
    table_simclr_summary = pd.read_csv(TABLE_SIMCLR_SUMMARY_DEP_PATH)

missing_categories = sorted(set(CATEGORIES_ACTIVE) - set(split_manifest.keys()))
if len(missing_categories) > 0:
    raise ValueError(
        f"The split manifest does not contain the required categories for this run: {missing_categories}"
    )

print_banner("Dataset and dependency check")
print("MVTEC_DIR                  :", MVTEC_DIR)
print("Split manifest path        :", SPLIT_MANIFEST_PATH)
print("Leakage report path        :", LEAKAGE_REPORT_PATH)
print("SSL checkpoints dir        :", SSL_CHECKPOINTS_DIR)
print("Available SSL checkpoints  :", available_ssl_tags)
print("SimCLR summary table found :", TABLE_SIMCLR_SUMMARY_DEP_PATH.exists())
if len(available_ssl_tags) < 2:
    print("Warning: only one SSL checkpoint is available, so augmentation-strength ablation will be limited.")
if table_simclr_summary is not None:
    display(table_simclr_summary)


# 4.0 Shared datasets, transforms, metrics, and plotting helpers

## Purpose
- define the shared dataset wrapper and dataloaders used by the ablation study
- keep evaluation logic consistent with the earlier downstream notebooks
- avoid reintroducing unrelated autoencoder or threshold-analysis code


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Shared transforms, dataset class, loaders, metrics, and plotting helpers
#------------------------------------------------------------------------------
tfm_imagenet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# Load one RGB image.
def load_rgb(path):
    return Image.open(path).convert("RGB")


# Load one binary mask or return an all-zero mask for good images.
def load_mask(path):
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

    mask = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(mask)[None, :, :]


# Return items in a consistent format across the shared split manifest.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if self.mode in ["train_good", "val_good"]:
            img_path = item
            label = 0
            mask_path = None
        else:
            img_path = item["img_path"]
            label = int(item["label"])
            mask_path = item.get("mask_path", None)

        image = self.img_tfm(load_rgb(img_path))
        mask = load_mask(mask_path)
        return image, int(label), mask, str(img_path)


# Build a DataLoader matched to the active runtime.
def make_loader(items, mode, batch_size, shuffle):
    dataset = MvtecDataset(items=items, mode=mode, img_tfm=tfm_imagenet)

    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE == "cuda",
    }

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**loader_kwargs)


# Build the train, validation, and test loaders for one category.
def make_split_loaders(category):
    train_loader = make_loader(split_manifest[category]["train_good"], mode="train_good", batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = make_loader(split_manifest[category]["val_good"], mode="val_good", batch_size=BATCH_SIZE_TEST, shuffle=False)
    test_loader = make_loader(split_manifest[category]["test"], mode="test", batch_size=BATCH_SIZE_TEST, shuffle=False)
    return train_loader, val_loader, test_loader


# Compute image-level ROC-AUC safely.
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Compute image-level PR-AUC safely.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


# Flatten masks and heatmaps to compute pixel ROC-AUC.
def pixel_roc_auc(masks_list, heatmaps_list):
    if len(masks_list) == 0 or len(heatmaps_list) == 0:
        return np.nan

    y_true = np.concatenate([mask.reshape(-1) for mask in masks_list]).astype(int)
    y_score = np.concatenate([heat.reshape(-1) for heat in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Evaluate one scoring function across the test set.
def eval_method(test_loader, score_fn):
    y_true = []
    y_score = []
    masks = []
    heats = []

    eval_start = time.time()
    eval_image_n = 0

    for images, labels, batch_masks, _ in test_loader:
        scores, heatmaps = score_fn(images)

        y_true.extend(labels.numpy().tolist())
        y_score.extend(scores.tolist())

        mask_np = batch_masks.squeeze(1).numpy()
        for idx in range(mask_np.shape[0]):
            masks.append(mask_np[idx])
            heats.append(heatmaps[idx])

        eval_image_n += images.shape[0]

    total_eval_sec = time.time() - eval_start

    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heats),
        "sec_per_img": total_eval_sec / max(eval_image_n, 1),
        "n_eval_images": int(eval_image_n),
    }


# 5.0 SSL runtime builders for ablations

## Purpose
- rebuild only the SSL downstream methods needed for the ablation study
- keep the evaluation deterministic across categories and settings
- isolate the design variables under test while reusing the same shared scoring logic


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Backbone, feature hooks, PatchCore, PaDiM, and SSL runtime builders
#------------------------------------------------------------------------------
def get_resnet18_ssl():
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Identity()
    model.eval()
    return model


# Register hooks so intermediate feature maps can be collected.
def make_feature_hooks(model, layer_names):
    features = {}
    handles = []

    def hook_factory(layer_name):
        def _hook(_, __, output):
            features[layer_name] = output
        return _hook

    for layer_name in layer_names:
        handle = getattr(model, layer_name).register_forward_hook(hook_factory(layer_name))
        handles.append(handle)

    return features, handles


# Remove hook handles after the runtime is no longer needed.
def remove_handles(handles):
    for handle in handles:
        handle.remove()


@torch.inference_mode()
def forward_get_feats(model, features, inputs, layer_names):
    _ = model(inputs)
    return [features[layer_name] for layer_name in layer_names]


# Flatten a feature map into patch rows.
def fmap_to_patches(feature_map):
    batch_n, channels, height, width = feature_map.shape
    return feature_map.permute(0, 2, 3, 1).reshape(batch_n, height * width, channels)


# Upsample and concatenate patch features from one or more layers.
def concat_patch_features(feature_maps):
    target_h, target_w = feature_maps[-1].shape[-2:]
    patch_list = []

    for feature_map in feature_maps:
        if feature_map.shape[-2:] != (target_h, target_w):
            feature_map = F.interpolate(
                feature_map,
                size=(target_h, target_w),
                mode="bilinear",
                align_corners=False,
            )
        patch_list.append(fmap_to_patches(feature_map))

    return torch.cat(patch_list, dim=-1)


# Build a FAISS L2 index for nearest-neighbour search.
def faiss_index_l2(array_2d):
    array_2d = np.asarray(array_2d, dtype=np.float32)
    index = faiss.IndexFlatL2(array_2d.shape[1])
    index.add(array_2d)
    return index


# Load a saved SimCLR encoder checkpoint and strip any DataParallel prefix if needed.
def load_ssl_encoder(ckpt_path):
    model = get_resnet18_ssl()
    state = torch.load(ckpt_path, map_location="cpu")

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if isinstance(state, dict):
        clean_state = {}
        for key, value in state.items():
            clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
            clean_state[clean_key] = value
        state = clean_state

    model.load_state_dict(state, strict=True)
    model.eval()
    return model


# Collect normal train patch embeddings and keep a deterministic coreset subset.
def build_patch_bank(model, features, train_loader, layer_names, category, aug_strength, coreset_ratio):
    patch_chunks = []
    total_patches = 0

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_maps = forward_get_feats(model, features, images, layer_names)
        patches = concat_patch_features(feature_maps).detach().cpu().numpy()
        patches = patches.reshape(-1, patches.shape[-1]).astype(np.float32)
        patch_chunks.append(patches)
        total_patches += patches.shape[0]

        if total_patches >= MAX_TRAIN_PATCHES:
            break

    bank_full = np.concatenate(patch_chunks, axis=0).astype(np.float32)

    if len(bank_full) > MAX_TRAIN_PATCHES:
        rng_cap = np.random.default_rng(
            stable_seed_from_text(SEED, f"{category}_{aug_strength}_{'_'.join(layer_names)}_patch_bank_cap")
        )
        idx_cap = rng_cap.choice(len(bank_full), size=MAX_TRAIN_PATCHES, replace=False)
        bank_full = bank_full[idx_cap]

    keep_n = max(1, int(round(len(bank_full) * float(coreset_ratio))))
    keep_n = min(keep_n, len(bank_full))

    rng_core = np.random.default_rng(
        stable_seed_from_text(SEED, f"{category}_{aug_strength}_{'_'.join(layer_names)}_{coreset_ratio}_patchcore_coreset")
    )
    coreset_idx = rng_core.choice(len(bank_full), size=keep_n, replace=False)
    bank = bank_full[coreset_idx]

    return bank


@torch.inference_mode()
def patchcore_scores(model, features, images, layer_names, index):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_maps = forward_get_feats(model, features, images, layer_names)
    patches = concat_patch_features(feature_maps).detach().cpu().numpy()

    batch_n, patch_n, feat_dim = patches.shape
    queries = patches.reshape(-1, feat_dim).astype(np.float32)
    dist2, _ = index.search(queries, 1)
    dist2 = dist2.reshape(batch_n, patch_n)

    feat_h, feat_w = feature_maps[-1].shape[-2:]
    heat = dist2.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Fit the PaDiM Gaussian statistics with a deterministic feature subset.
def padim_fit(model, features, train_loader, layer_name, category, aug_strength):
    all_feature_maps = []

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()
        all_feature_maps.append(feature_map)

    feature_map = torch.cat(all_feature_maps, dim=0)
    sample_n, channels, feat_h, feat_w = feature_map.shape

    dim = min(PADIM_DIM, channels)
    rng_dim = np.random.default_rng(
        stable_seed_from_text(SEED, f"{category}_{aug_strength}_{layer_name}_padim_dim")
    )
    dim_idx = rng_dim.choice(channels, size=dim, replace=False)

    feature_map = feature_map[:, dim_idx, :, :]
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(sample_n, feat_h * feat_w, dim).numpy()

    mu = feats_np.mean(axis=0)
    cov_inv = []

    eye = np.eye(dim, dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        patch_features = feats_np[:, patch_idx, :]
        cov = np.cov(patch_features, rowvar=False).astype(np.float32) + PADIM_EPS * eye
        cov_inv.append(np.linalg.inv(cov).astype(np.float32))

    cov_inv = np.stack(cov_inv, axis=0)

    return {
        "dim_idx": dim_idx.astype(np.int64),
        "mu": mu.astype(np.float32),
        "cov_inv": cov_inv.astype(np.float32),
        "H": int(feat_h),
        "W": int(feat_w),
        "D": int(dim),
    }


@torch.inference_mode()
def padim_scores(model, features, images, layer_name, stats):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()

    dim_idx = torch.tensor(stats["dim_idx"])
    feature_map = feature_map[:, dim_idx, :, :]

    batch_n, feat_dim, feat_h, feat_w = feature_map.shape
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(batch_n, feat_h * feat_w, feat_dim).numpy()

    mu = stats["mu"]
    cov_inv = stats["cov_inv"]

    heat = np.zeros((batch_n, feat_h * feat_w), dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        diff = feats_np[:, patch_idx, :] - mu[patch_idx]
        tmp = diff @ cov_inv[patch_idx]
        heat[:, patch_idx] = np.sum(tmp * diff, axis=1)

    heat = heat.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Rebuild one SSL downstream runtime for the requested ablation setting.
def build_ssl_runtime(category, aug_strength, anomaly_head, layer_name, coreset_ratio):
    train_loader, val_loader, test_loader = make_split_loaders(category)

    ckpt_path = Path(available_ssl_ckpt_map[aug_strength])
    model = load_ssl_encoder(ckpt_path).to(DEVICE)
    features, handles = make_feature_hooks(model, [layer_name])

    reset_peak_gpu_memory()
    fit_start = time.time()

    if anomaly_head == "patchcore":
        bank = build_patch_bank(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_names=[layer_name],
            category=category,
            aug_strength=aug_strength,
            coreset_ratio=coreset_ratio,
        )
        index = faiss_index_l2(bank)

        def score_fn(images):
            return patchcore_scores(model, features, images, [layer_name], index)

        fit_stats = {
            "feature_dim": int(bank.shape[1]),
            "memory_bank_n": int(bank.shape[0]),
            "memory_bank_mb": float(bank.nbytes / (1024 ** 2)),
        }

    elif anomaly_head == "padim":
        stats = padim_fit(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_name=layer_name,
            category=category,
            aug_strength=aug_strength,
        )

        def score_fn(images):
            return padim_scores(model, features, images, layer_name, stats)

        fit_stats = {
            "feature_dim": int(stats["D"]),
            "memory_bank_n": int(stats["H"] * stats["W"]),
            "memory_bank_mb": float((stats["mu"].nbytes + stats["cov_inv"].nbytes) / (1024 ** 2)),
        }

    else:
        remove_handles(handles)
        raise ValueError(f"Unsupported anomaly head: {anomaly_head}")

    fit_sec = float(time.time() - fit_start)
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "features": features,
        "handles": handles,
        "score_fn": score_fn,
        "feature_dim": fit_stats["feature_dim"],
        "memory_bank_n": fit_stats["memory_bank_n"],
        "memory_bank_mb": fit_stats["memory_bank_mb"],
        "fit_sec": fit_sec,
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)),
    }


# 6.0 Run the ablation study

## Purpose
- vary augmentation strength while holding the downstream setup fixed
- vary the embedding layer while keeping the chosen SSL checkpoint and anomaly head fixed
- vary the PatchCore coreset ratio so the accuracy versus memory trade-off can be seen directly


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Run the ablation loops and save tables
#------------------------------------------------------------------------------
ABLATION_SSL_TAG = "strong" if "strong" in available_ssl_tags else available_ssl_tags[0]

def run_ssl_ablation_category(category, aug_strength, anomaly_head, layer_name, coreset_ratio):
    runtime = build_ssl_runtime(
        category=category,
        aug_strength=aug_strength,
        anomaly_head=anomaly_head,
        layer_name=layer_name,
        coreset_ratio=coreset_ratio,
    )
    res = eval_method(runtime["test_loader"], runtime["score_fn"])

    row = {
        "category": category,
        "representation_source": "simclr",
        "anomaly_head": anomaly_head,
        "aug_strength": aug_strength,
        "layer_name": layer_name,
        "coreset_ratio": float(coreset_ratio) if not pd.isna(coreset_ratio) else np.nan,
        "feature_dim": runtime["feature_dim"],
        "memory_bank_n": runtime["memory_bank_n"],
        "memory_bank_mb": runtime["memory_bank_mb"],
        "checkpoint_size_mb": runtime["checkpoint_size_mb"],
        "fit_sec": runtime["fit_sec"],
        "peak_gpu_mem_mb": runtime["peak_gpu_mem_mb"],
        **res,
    }

    remove_handles(runtime["handles"])
    del runtime["model"]
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return row


ablation_rows = []

# A. augmentation strength
print_banner("Ablation A: augmentation strength")
for aug_strength in available_ssl_tags:
    for category in CATEGORIES_ACTIVE:
        ablation_rows.append({
            "ablation_factor": "augmentation_strength",
            "setting": aug_strength,
            **run_ssl_ablation_category(
                category=category,
                aug_strength=aug_strength,
                anomaly_head="patchcore",
                layer_name=FEATURE_LAYER_NAME_MAIN,
                coreset_ratio=CORESET_RATIO_MAIN,
            ),
        })
    for category in CATEGORIES_ACTIVE:
        ablation_rows.append({
            "ablation_factor": "augmentation_strength_padim",
            "setting": aug_strength,
            **run_ssl_ablation_category(
                category=category,
                aug_strength=aug_strength,
                anomaly_head="padim",
                layer_name=FEATURE_LAYER_NAME_MAIN,
                coreset_ratio=np.nan,
            ),
        })

# B. embedding layer
print_banner("Ablation B: embedding layer")
for layer_name in LAYER_ABL_VALUES:
    for category in CATEGORIES_ACTIVE:
        ablation_rows.append({
            "ablation_factor": "embedding_layer",
            "setting": layer_name,
            **run_ssl_ablation_category(
                category=category,
                aug_strength=ABLATION_SSL_TAG,
                anomaly_head="patchcore",
                layer_name=layer_name,
                coreset_ratio=CORESET_RATIO_MAIN,
            ),
        })

# C. PatchCore coreset ratio
print_banner("Ablation C: coreset ratio")
for ratio in CORESET_ABL_VALUES:
    for category in CATEGORIES_ACTIVE:
        ablation_rows.append({
            "ablation_factor": "coreset_ratio",
            "setting": str(ratio),
            **run_ssl_ablation_category(
                category=category,
                aug_strength=ABLATION_SSL_TAG,
                anomaly_head="patchcore",
                layer_name=FEATURE_LAYER_NAME_MAIN,
                coreset_ratio=ratio,
            ),
        })

df_ablation_category = pd.DataFrame(ablation_rows)

mean_cols = [
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
    "n_eval_images",
]
df_ablation_mean = (
    df_ablation_category
    .groupby(
        [
            "ablation_factor",
            "setting",
            "representation_source",
            "anomaly_head",
            "aug_strength",
            "layer_name",
            "coreset_ratio",
        ],
        as_index=False
    )[mean_cols]
    .mean(numeric_only=True)
)
df_ablation_mean["category"] = "MEAN"

ordered_cols = [
    "ablation_factor",
    "setting",
    "category",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "layer_name",
    "coreset_ratio",
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
    "n_eval_images",
]
df_ablation_category = df_ablation_category[ordered_cols]
df_ablation_mean = df_ablation_mean[ordered_cols]
df_ablation_full = pd.concat([df_ablation_category, df_ablation_mean], ignore_index=True)

save_csv_overwrite(df_ablation_category, TABLE_ABLATION_CATEGORY_PATH)
save_csv_overwrite(df_ablation_mean, TABLE_ABLATION_MEAN_PATH)
save_csv_overwrite(df_ablation_full, TABLE_ABLATION_FULL_PATH)

print_banner("Ablation sanity checks")
aug_check = df_ablation_mean[df_ablation_mean["ablation_factor"] == "augmentation_strength"][["setting", "img_pr_auc", "img_roc_auc", "px_roc_auc"]].copy()
aug_padim_check = df_ablation_mean[df_ablation_mean["ablation_factor"] == "augmentation_strength_padim"][["setting", "img_pr_auc", "img_roc_auc", "px_roc_auc"]].copy()
layer_check = df_ablation_mean[df_ablation_mean["ablation_factor"] == "embedding_layer"][["setting", "img_pr_auc", "img_roc_auc", "px_roc_auc"]].copy()
coreset_check = df_ablation_mean[df_ablation_mean["ablation_factor"] == "coreset_ratio"][["setting", "memory_bank_n", "memory_bank_mb", "img_pr_auc"]].copy()

print("\nPatchCore augmentation summary")
display(aug_check.sort_values("setting"))

print("\nPaDiM augmentation summary")
display(aug_padim_check.sort_values("setting"))

print("\nLayer summary")
display(layer_check.sort_values("setting"))

print("\nCoreset summary")
display(coreset_check.sort_values("setting"))

print("Saved category table:", TABLE_ABLATION_CATEGORY_PATH)
print("Saved mean table    :", TABLE_ABLATION_MEAN_PATH)
print("Saved full table    :", TABLE_ABLATION_FULL_PATH)


# 7.0 Ablation figures

## Purpose
- turn the ablation tables into a compact figure that is easy to skim in GitHub and in the report
- show both the performance effects and the coreset memory trade-off in one place


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Save the ablation summary figure
#------------------------------------------------------------------------------
fig = plt.figure(figsize=(16, 4.8))

# Panel A: augmentation strength
ax1 = plt.subplot(1, 3, 1)
plot_aug = df_ablation_mean[
    df_ablation_mean["ablation_factor"].isin(["augmentation_strength", "augmentation_strength_padim"])
].copy()
if len(plot_aug) > 0:
    plot_aug["series"] = plot_aug["anomaly_head"].str.upper()
    x_order = sorted(plot_aug["setting"].dropna().unique().tolist())
    x = np.arange(len(x_order))
    width = 0.35 if plot_aug["series"].nunique() > 1 else 0.55

    for idx, series_name in enumerate(sorted(plot_aug["series"].unique().tolist())):
        tmp = plot_aug[plot_aug["series"] == series_name].copy()
        tmp = tmp.set_index("setting").reindex(x_order)
        offset = (idx - (tmp["series"].nunique() - 1) / 2.0) * width
        values = tmp["img_pr_auc"].values.astype(float)
        bars = ax1.bar(x + (idx - (len(sorted(plot_aug["series"].unique().tolist())) - 1) / 2.0) * width, values, width=width, label=series_name)
        for bar, value in zip(bars, values):
            if not np.isnan(value):
                ax1.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)

    ax1.set_xticks(x)
    ax1.set_xticklabels(x_order)
    ax1.set_ylim(0, 1.05)
    ax1.set_title("A. Augmentation strength")
    ax1.set_ylabel("Mean image PR-AUC")
    ax1.legend()

# Panel B: embedding layer
ax2 = plt.subplot(1, 3, 2)
plot_layer = df_ablation_mean[df_ablation_mean["ablation_factor"] == "embedding_layer"].copy()
if len(plot_layer) > 0:
    layer_order = [layer for layer in LAYER_ABL_VALUES if layer in plot_layer["setting"].tolist()]
    vals = plot_layer.set_index("setting").reindex(layer_order)["img_pr_auc"].values.astype(float)
    ax2.plot(layer_order, vals, marker="o")
    for x_val, y_val in zip(layer_order, vals):
        if not np.isnan(y_val):
            ax2.text(x_val, y_val, f"{y_val:.3f}", ha="center", va="bottom", fontsize=8)
    ax2.set_ylim(0, 1.05)
    ax2.set_title("B. Embedding layer")
    ax2.set_ylabel("Mean image PR-AUC")
    ax2.grid(True, alpha=0.3)

# Panel C: coreset ratio and memory trade-off
ax3 = plt.subplot(1, 3, 3)
plot_core = df_ablation_mean[df_ablation_mean["ablation_factor"] == "coreset_ratio"].copy()
if len(plot_core) > 0:
    plot_core["setting_float"] = plot_core["setting"].astype(float)
    plot_core = plot_core.sort_values("setting_float")

    ax3.plot(plot_core["setting_float"].values, plot_core["img_pr_auc"].values, marker="o")
    for x_val, y_val in zip(plot_core["setting_float"].values, plot_core["img_pr_auc"].values):
        if not np.isnan(y_val):
            ax3.text(x_val, y_val, f"{y_val:.3f}", ha="center", va="bottom", fontsize=8)

    ax3.set_xlabel("Coreset ratio")
    ax3.set_ylabel("Mean image PR-AUC")
    ax3.set_title("C. Coreset ratio trade-off")
    ax3.grid(True, alpha=0.3)

    ax3b = ax3.twinx()
    ax3b.plot(plot_core["setting_float"].values, plot_core["memory_bank_n"].values, marker="s", linestyle="--")
    ax3b.set_ylabel("Mean memory bank size")

plt.tight_layout()
ensure_parent(FIG_ABLATION_SUMMARY_PATH)
plt.savefig(FIG_ABLATION_SUMMARY_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_ABLATION_SUMMARY_PATH)


# 8.0 Final review

## Purpose
- print the main saved artefacts for quick checking
- surface the mean ablation table at the end of the notebook for easy review


In [ ]:
#------------------------------------------------------------------------------
# 8.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata path           :", RUN_METADATA_PATH)
print("Ablation settings path      :", ABLATION_SETTINGS_PATH)
print("Available SSL checkpoints   :", AVAILABLE_SSL_CHECKPOINTS_PATH)
print("Ablation category table     :", TABLE_ABLATION_CATEGORY_PATH)
print("Ablation mean table         :", TABLE_ABLATION_MEAN_PATH)
print("Ablation full table         :", TABLE_ABLATION_FULL_PATH)
print("Ablation summary figure     :", FIG_ABLATION_SUMMARY_PATH)

display(df_ablation_mean.sort_values(["ablation_factor", "setting"]).reset_index(drop=True))
